# Character-level RNN language model → sampling from Llama

This notebook has two parts:

1. **Char-RNN.** Train a character-level LSTM language model on **Tiny Shakespeare** (Karpathy's public char-rnn dataset, about 1 MB of text). Then sample from it with temperature, top-k, nucleus (top-p) sampling and beam search.
2. **Llama.** Load a pretrained Llama model with 🤗 `transformers`, and run the same decoding strategies on it. First use `model.generate`, then write them **from scratch** with a KV cache. At the end, compare the two models by bits per character on the same held-out text.

The decoding utilities (temperature / top-k / nucleus) are written once and shared by both models: an LM is an LM, whether its tokens are characters or BPE pieces.

**Runtime:** use a GPU (Colab: *Runtime → Change runtime type → T4 GPU*). Part 1 also runs on CPU if you reduce `N_STEPS`.

In [ ]:
import os, math, random, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import trange
from IPython.display import clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything()

## Sampling toolkit (shared by both parts)

At every step a language model gives us logits $z \in \mathbb{R}^{|V|}$ for the next token. The strategies below differ only in how they turn $z$ into a choice:

| strategy | rule |
|---|---|
| greedy | $\arg\max_i z_i$ |
| temperature $\tau$ | sample from $\mathrm{softmax}(z/\tau)$. $\tau\to 0$ gives greedy; $\tau>1$ flattens the distribution |
| top-k | keep the $k$ most likely tokens, renormalize, sample |
| nucleus / top-p | keep the **smallest** set of most likely tokens whose total mass is $\ge p$, renormalize, sample |

> **Note on the nucleus definition.** Holtzman et al. (2019) and Hugging Face keep the token that pushes the cumulative mass past $p$. Some course notebooks instead keep only tokens while the cumulative mass stays *below* $p$. For $p=[0.1, 0.35, 0.05, 0.2, 0.3]$ and $p_{nuc}=0.9$, the first rule keeps 4 tokens and the second keeps 3. We use the standard rule, so our results match `generate(top_p=...)`.

In [ ]:
def top_k_filter(probs: torch.Tensor, k: int) -> torch.Tensor:
    """Zero out everything except the k most likely entries; renormalize. probs: [V]"""
    k = min(k, probs.numel())
    vals, idx = torch.topk(probs, k)
    out = torch.zeros_like(probs)
    out[idx] = vals
    return out / out.sum()

def nucleus_filter(probs: torch.Tensor, p: float) -> torch.Tensor:
    """Keep the smallest prefix of the sorted distribution with mass >= p (always >= 1 token)."""
    sorted_p, idx = torch.sort(probs, descending=True)
    mass_before = torch.cumsum(sorted_p, dim=0) - sorted_p   # mass of strictly-more-likely tokens
    keep = mass_before < p                                   # first token always kept
    out = torch.zeros_like(probs)
    out[idx[keep]] = sorted_p[keep]
    return out / out.sum()

def pick_next(logits: torch.Tensor, temperature=1.0, top_k=None, top_p=None) -> int:
    """Choose the next token id from a 1-D logits vector."""
    logits = logits.float()
    if temperature == 0:
        return int(logits.argmax())
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_k is not None:
        probs = top_k_filter(probs, top_k)
    if top_p is not None:
        probs = nucleus_filter(probs, top_p)
    return int(torch.multinomial(probs, num_samples=1))

# sanity checks on the toy example
p = torch.tensor([0.1, 0.35, 0.05, 0.2, 0.3])
print("top-3   :", top_k_filter(p, 3).numpy().round(3))      # [0, .412, 0, .235, .353]
print("top-p .9:", nucleus_filter(p, 0.9).numpy().round(3))  # w2, w5, w4 and w1 (mass before w1 = 0.85 < 0.9)
assert torch.allclose(top_k_filter(p, 3), torch.tensor([0, .35, 0, .2, .3]) / .85)
assert (nucleus_filter(p, 0.9) > 0).sum() == 4
assert (nucleus_filter(p, 0.01) > 0).sum() == 1             # nucleus is never empty

# Part 1 — Character-level RNN language model

### Data: Tiny Shakespeare
We split the text into **speeches**: blocks separated by blank lines, such as `"ROMEO:\nBut soft! ..."`. Each speech becomes one training sequence, wrapped in special tokens:

* **BOS** (begin of sequence) gives the network non-empty input, so $P(x_1) = P(x_1\mid \text{BOS})$.
* **EOS** (end of sequence) appears only at the end. When the model emits it, the sample is over.

Shakespeare uses `\n` inside speeches, so `\n` can't serve as EOS. We use two non-printing ASCII control characters instead. Long speeches are cut **at a line boundary** to at most `MAX_CHARS` characters, so the model never learns to stop in the middle of a word.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
PATH = "tinyshakespeare.txt"
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
text = open(PATH, encoding="utf-8").read()
print(f"{len(text):,} characters")
print(text[:400])

In [ ]:
BOS, EOS = "\x02", "\x03"     # STX / ETX control chars: never occur in the text
MAX_CHARS = 500

def cut_at_line(block, max_chars=MAX_CHARS):
    out = ""
    for line in block.split("\n"):
        if len(out) + len(line) + 1 > max_chars and out:
            break
        out += ("\n" if out else "") + line
    return out[:max_chars]

assert BOS not in text and EOS not in text
speeches = [cut_at_line(b.strip()) for b in text.split("\n\n") if b.strip()]
lines = [BOS + s + EOS for s in speeches]

random.Random(0).shuffle(lines)
n_dev = len(lines) // 10
dev_lines, train_lines = lines[:n_dev], lines[n_dev:]
print(f"{len(lines)} speeches | train {len(train_lines)} | dev {len(dev_lines)}")
print("mean length:", np.mean([len(l) for l in lines]).round(1), "| max:", max(map(len, lines)))

def show(s):  # strip special tokens for printing
    return s.replace(BOS, "").replace(EOS, "")
print("-" * 40); print(show(train_lines[0]))

### Vocabulary and batching

In [ ]:
tokens = sorted(set("".join(lines)))
n_tokens = len(tokens)
token_to_id = {t: i for i, t in enumerate(tokens)}
print("n_tokens =", n_tokens)
print(repr("".join(tokens)))
assert BOS in token_to_id and EOS in token_to_id

def to_matrix(lines, max_len=None, pad=token_to_id[EOS], dtype=np.int64):
    """Cast a list of strings into an int matrix [batch, max_len], padded with EOS."""
    max_len = max_len or max(map(len, lines))
    lines_ix = np.full([len(lines), max_len], pad, dtype=dtype)
    for i, line in enumerate(lines):
        line_ix = [token_to_id[c] for c in line[:max_len]]
        lines_ix[i, :len(line_ix)] = line_ix
    return lines_ix

dummy_lines = [BOS + "abc" + EOS, BOS + "abacaba" + EOS, BOS + "To be, or not" + EOS]
print(to_matrix(dummy_lines))

### Loss with masking

$$ L = \frac{1}{\#\text{tokens}}\sum_{X\in D}\sum_{t} -\log p(x_t\mid x_{<t},\theta) $$

Only real tokens count, **including the first EOS**. The padding EOS tokens after it must not contribute to the loss.

In [ ]:
def compute_mask(input_ix, eos_ix=token_to_id[EOS]):
    """True up to and including the first EOS."""
    return F.pad(torch.cumsum(input_ix == eos_ix, dim=-1)[..., :-1] < 1, pad=(1, 0, 0, 0), value=True)

def compute_loss(model, input_ix):
    """Mean cross-entropy over real (non-padding) target tokens."""
    input_ix = torch.as_tensor(input_ix, dtype=torch.int64, device=device)
    logits, _ = model(input_ix[:, :-1])                 # [B, T-1, V]
    targets = input_ix[:, 1:]                           # [B, T-1]
    mask = compute_mask(input_ix)[:, 1:].float()        # mask on *targets*
    nll = F.cross_entropy(logits.transpose(1, 2), targets, reduction="none")  # [B, T-1]
    return (nll * mask).sum() / mask.sum()

m = compute_mask(torch.as_tensor(to_matrix(dummy_lines)))
print(m.int().numpy()); print("lengths:", m.sum(-1).numpy())

### Model

$$ h_0=\vec 0;\quad h_{t}=\mathrm{LSTM}(e(x_t),h_{t-1});\quad p(x_{t+1}\mid x_{\le t})=\mathrm{softmax}(W h_t + b) $$

`forward` returns `(logits, state)`. Passing `state` back in lets us generate one character at a time without re-reading the prefix. This is the RNN analogue of a transformer's KV cache, which we use in Part 2.

In [ ]:
class CharRNNLM(nn.Module):
    def __init__(self, n_tokens=n_tokens, emb_size=64, hid_size=512, num_layers=2, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_tokens, emb_size)
        self.rnn = nn.LSTM(emb_size, hid_size, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hid_size, n_tokens)

    def forward(self, input_ix, state=None):
        """input_ix: int64 [batch, length] -> logits [batch, length, n_tokens], new state"""
        h, state = self.rnn(self.emb(input_ix), state)
        return self.out(self.drop(h)), state

model = CharRNNLM().to(device)
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# shape / causality tests (eval mode -> dropout off, deterministic)
model.eval()
x1 = torch.as_tensor(to_matrix(dummy_lines), device=device)
x2 = torch.as_tensor(to_matrix([l[:3] + "e" * (len(l) - 3) for l in dummy_lines]), device=device)
with torch.no_grad():
    l1, _ = model(x1); l2, _ = model(x2)
assert l1.shape == (len(dummy_lines), max(map(len, dummy_lines)), n_tokens)
assert torch.allclose(l1[:, :3], l2[:, :3], atol=1e-5), "model looks into the future!"
assert not torch.allclose(l1.sum(-1), torch.ones_like(l1.sum(-1))), "return logits, not probabilities"

loss_1 = compute_loss(model, to_matrix(dummy_lines, max_len=16))
loss_2 = compute_loss(model, to_matrix(dummy_lines, max_len=20))
assert torch.allclose(loss_1, loss_2), "padding leaks into the loss"
print("untrained loss: %.3f  (uniform would be ln|V| = %.3f)" % (loss_1.item(), math.log(n_tokens)))

### Evaluation and sampling helpers

We report **bits per character**, $\text{bpc} = L / \ln 2$. A uniform model over the vocabulary scores $\log_2|V|\approx 6$. A good char-RNN on this data gets about 1.5 bpc.

In [ ]:
@torch.no_grad()
def score_lines(model, lines, batch_size=128):
    """Token-weighted mean NLL (nats/char) over a list of lines."""
    model.eval()
    total, count = 0.0, 0
    for i in range(0, len(lines), batch_size):
        batch = torch.as_tensor(to_matrix(lines[i:i + batch_size]), device=device)
        n = compute_mask(batch)[:, 1:].sum().item()
        total += compute_loss(model, batch).item() * n
        count += n
    return total / count

@torch.no_grad()
def generate(model, prefix=BOS, max_len=600, **pick_kwargs):
    """Sample one character at a time, carrying the LSTM state (no re-encoding of the prefix)."""
    model.eval()
    if not prefix.startswith(BOS):
        prefix = BOS + prefix
    ix = torch.as_tensor([[token_to_id[c] for c in prefix]], device=device)
    logits, state = model(ix)
    out = prefix
    while len(out) < max_len:
        nxt = pick_next(logits[0, -1], **pick_kwargs)
        out += tokens[nxt]
        if tokens[nxt] == EOS:
            break
        logits, state = model(torch.tensor([[nxt]], device=device), state)
    return out

print("Before training:\n", show(generate(model, "ROMEO:\n", max_len=120)))

### Training

This is the same objective as the fixed-window model. RNNs multiply gradients through hundreds of time steps, so we clip the global gradient norm. We also use a cosine learning-rate schedule.

On a T4, 3000 steps take a few minutes. On CPU, use `N_STEPS = 500` and `hid_size=256`.

In [ ]:
N_STEPS     = 3000
BATCH_SIZE  = 64
LR          = 2e-3
CLIP        = 1.0
EVAL_EVERY  = 250

seed_everything(0)
model = CharRNNLM().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_STEPS, eta_min=LR / 20)
train_history, dev_history = [], [(0, score_lines(model, dev_lines))]

for step in trange(N_STEPS):
    model.train()
    batch = to_matrix(random.sample(train_lines, BATCH_SIZE))
    loss = compute_loss(model, batch)
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    opt.step(); sched.step()
    train_history.append((step, loss.item()))

    if (step + 1) % EVAL_EVERY == 0:
        dev_history.append((step + 1, score_lines(model, dev_lines)))
        clear_output(wait=True)
        s, l = zip(*train_history)
        plt.figure(figsize=(8, 3.5))
        plt.plot(s, np.array(l) / math.log(2), alpha=0.3, label="train")
        s, l = zip(*dev_history)
        plt.plot(s, np.array(l) / math.log(2), "o-", color="red", label="dev")
        plt.ylabel("bits / char"); plt.xlabel("step"); plt.grid(alpha=.3); plt.legend(); plt.show()
        print(f"step {step + 1}: dev {dev_history[-1][1] / math.log(2):.3f} bpc")
        print(show(generate(model, max_len=250, temperature=0.7)))

In [ ]:
assert np.mean([l for _, l in train_history[:10]]) > np.mean([l for _, l in train_history[-10:]]), "didn't converge"
char_rnn_dev_bpc = dev_history[-1][1] / math.log(2)
print(f"Final dev: {char_rnn_dev_bpc:.3f} bits/char")
torch.save(model.state_dict(), "char_rnn_shakespeare.pt")

### Sampling strategies on the char-RNN

Compare the outputs:

* **low temperature** is fluent but repetitive;
* **high temperature** is diverse, but invents nonsense words;
* **top-k / nucleus** remove the long tail of unlikely characters while keeping the diversity.

In [ ]:
settings = {
    "greedy (τ=0)":       dict(temperature=0),
    "τ = 0.5":            dict(temperature=0.5),
    "τ = 1.0":            dict(temperature=1.0),
    "τ = 1.5":            dict(temperature=1.5),
    "top-k = 5":          dict(top_k=5),
    "nucleus p = 0.9":    dict(top_p=0.9),
    "nucleus p = 0.5":    dict(top_p=0.5),
}
seed_everything(1)
for name, kw in settings.items():
    print(f"===== {name} =====")
    print(show(generate(model, "KING HENRY:\n", max_len=300, **kw)), "\n")

In [ ]:
# How big is the nucleus, character by character? It's tiny inside a word and large after a space.
@torch.no_grad()
def nucleus_sizes(model, s, p=0.9):
    ix = torch.as_tensor([[token_to_id[c] for c in BOS + s]], device=device)
    model.eval(); logits, _ = model(ix)
    probs = torch.softmax(logits[0], -1)
    return [(nucleus_filter(pr, p) > 0).sum().item() for pr in probs[:-1]]

s = "ROMEO:\nWhat light through yonder window breaks?"
sizes = nucleus_sizes(model, s)
plt.figure(figsize=(12, 2.5))
plt.bar(range(len(s)), sizes)
plt.xticks(range(len(s)), [c if c != "\n" else "⏎" for c in s])
plt.ylabel("|nucleus| (p=0.9)"); plt.title("size of nucleus for predicting each character"); plt.show()

### Beam search (char-RNN)

Beam search looks for a single **most likely** continuation rather than a diverse sample. At each step it expands every hypothesis by every token, then keeps the `beam_size` best by total log-probability. Hypotheses that emitted EOS are frozen.

The LSTM states are reordered with the chosen beam indices, so all beams run as one batch.

In [ ]:
@torch.no_grad()
def beam_search(model, prefix=BOS, beam_size=4, length=60):
    """Returns [(text, logprob)] sorted best-first. `length` = max new characters."""
    model.eval()
    if not prefix.startswith(BOS):
        prefix = BOS + prefix
    eos = token_to_id[EOS]
    ix = torch.as_tensor([[token_to_id[c] for c in prefix]], device=device)
    logits, state = model(ix)
    logp = torch.log_softmax(logits[:, -1], -1)                 # [B=1, V]
    texts, done = [prefix], [False]
    scores = torch.zeros(1, device=device)

    for _ in range(length):
        cand = scores[:, None] + logp                             # [B, V]
        for b, d in enumerate(done):                              # finished beams: one candidate = itself
            if d:
                cand[b] = float("-inf"); cand[b, eos] = scores[b]
        scores, flat = cand.view(-1).topk(min(beam_size, cand.numel()))
        b_ix, t_ix = flat // n_tokens, flat % n_tokens
        new_texts, new_done = [], []
        for b, t in zip(b_ix.tolist(), t_ix.tolist()):
            if done[b]:
                new_texts.append(texts[b]); new_done.append(True)
            else:
                new_texts.append(texts[b] + tokens[t]); new_done.append(t == eos)
        texts, done = new_texts, new_done
        if all(done):
            break
        state = tuple(s[:, b_ix] for s in state)                  # reorder LSTM state to follow beams
        logits, state = model(t_ix[:, None], state)
        logp = torch.log_softmax(logits[:, -1], -1)
    return [(show(t), s) for t, s in zip(texts, scores.tolist())]

for k in (1, 4, 8):
    best, lp = beam_search(model, "ROMEO:\n", beam_size=k, length=80)[0]
    print(f"--- beam {k}  (log p = {lp:.1f})\n{best}\n")

In [ ]:
# Exercise: find prefixes where beam_size=1 and beam_size=8 disagree
for pref in ["First Citizen:\n", "JULIET:\nO ", "KING RICHARD III:\n", "Second Murderer:\n", "LADY ANNE:\nThou ", "GLOUCESTER:\nNow is the "]:
    a = beam_search(model, pref, beam_size=1, length=50)[0][0]
    b = beam_search(model, pref, beam_size=8, length=50)[0][0]
    print(f"{'DIFFERENT' if a != b else 'same     '} | {a!r}\n            {b!r}\n")

# Part 2 — Sampling from Llama

We now switch from a 3M-parameter character model to a pretrained **Llama 3.2** model. Its vocabulary has 128k byte-level BPE tokens instead of 67 characters, but the next-token interface is the same, so `pick_next`, `top_k_filter` and `nucleus_filter` from the toolkit work unchanged.

**Access.** `meta-llama/*` repositories are **gated**. Accept the license on the model page, then log in with a token (`huggingface_hub.login()` or a Colab secret named `HF_TOKEN`). If you can't log in, the cell below falls back to an ungated mirror of the same weights.

| model | fp16 size | fits a T4 (16 GB)? |
|---|---|---|
| `Llama-3.2-1B` | ~2.5 GB | yes (default) |
| `Llama-3.2-3B` | ~6.5 GB | yes |
| `Llama-3.1-8B` | ~16 GB | only in 4-bit |

We use the **base** (non-instruct) models: we want a raw LM to sample from, not a chat assistant.

> If GPU memory is tight, you can restart the kernel here. Before continuing, re-run the **imports** cell and the **sampling toolkit** cell. The comparison at the very end also needs `dev_lines` and the char-RNN, so either keep the kernel or reload `char_rnn_shakespeare.pt`.

In [ ]:
# Colab usually has recent versions already; uncomment to upgrade.
# %pip install -q -U transformers accelerate
# %pip install -q -U bitsandbytes          # only for 4-bit loading
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
print("transformers", transformers.__version__)

# Optional: log in for gated meta-llama repos
# from huggingface_hub import login; login()

In [ ]:
CANDIDATES = [
    "meta-llama/Llama-3.2-1B",   # official, gated
    "unsloth/Llama-3.2-1B",      # ungated mirror of the same base weights
]
# Bigger: "meta-llama/Llama-3.2-3B"; 8B needs 4-bit (see below).

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4 -> float16
else:
    dtype = torch.float32

load_kwargs = dict(dtype=dtype, device_map="auto")   # on transformers < 4.56 use torch_dtype=dtype
# 4-bit alternative (pip install bitsandbytes):
# from transformers import BitsAndBytesConfig
# load_kwargs = dict(device_map="auto", quantization_config=BitsAndBytesConfig(
#     load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))

for model_name in CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        llm = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        break
    except Exception as e:
        print(f"could not load {model_name}: {type(e).__name__}: {str(e)[:150]}")
else:
    raise RuntimeError("no model could be loaded")

llm.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"loaded {model_name}: {sum(p.numel() for p in llm.parameters()) / 1e9:.2f}B params, "
      f"vocab {len(tokenizer):,}, dtype {next(llm.parameters()).dtype}, device {llm.device}")

### Tokenization

Unlike the char-RNN, Llama sees sub-word pieces. In the byte-level BPE display, `Ġ` marks a leading space. The tokenizer also prepends `<|begin_of_text|>`, which plays the role of our BOS.

In [ ]:
enc = tokenizer("Moscow is the capital of Russia. Elbrus is the highest mountain.")
print(enc.input_ids)
print(tokenizer.convert_ids_to_tokens(enc.input_ids))
print(repr(tokenizer.decode(enc.input_ids)))

### Decoding with `model.generate`

See the Hugging Face [generation strategies guide](https://huggingface.co/docs/transformers/generation_strategies). Base Llama checkpoints ship a `generation_config` with default `temperature`/`top_p`, so each strategy below sets its arguments explicitly.

In [ ]:
prompt = "The first discovered martian lifeform looks like"
batch = tokenizer(prompt, return_tensors="pt").to(llm.device)

strategies = {
    "greedy":                 dict(do_sample=False),
    "beam search (4)":        dict(do_sample=False, num_beams=4, early_stopping=True),
    "sampling τ=0.7":         dict(do_sample=True, temperature=0.7, top_k=0, top_p=1.0),
    "sampling τ=1.3":         dict(do_sample=True, temperature=1.3, top_k=0, top_p=1.0),
    "top-k=20":               dict(do_sample=True, temperature=1.0, top_k=20, top_p=1.0),
    "nucleus p=0.9":          dict(do_sample=True, temperature=1.0, top_k=0, top_p=0.9),
}
for name, kw in strategies.items():
    torch.manual_seed(1337)
    with torch.no_grad():
        out = llm.generate(**batch, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id, **kw)
    print(f"===== {name}\n{tokenizer.decode(out[0][batch.input_ids.shape[1]:], skip_special_tokens=True)}\n")

### Low-level generation with a KV cache

This is what `generate` does internally. The first forward pass encodes the whole prompt and returns `past_key_values`, the cached keys and values of every attention layer. Each later step feeds **only the new token** together with the cache, so the cost per step is $O(1)$ in the prompt length instead of re-running the whole sequence. Compare this with carrying the LSTM `state` in Part 1.

In [ ]:
@torch.no_grad()
def show_steps(prompt, steps=10, top=5, **pick_kwargs):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(llm.device)
    out = llm(input_ids=ids, use_cache=True)
    past, logits = out.past_key_values, out.logits[0, -1]
    print(prompt)
    for i in range(steps):
        probs = torch.softmax(logits.float(), -1)
        tp, ti = probs.topk(top)
        nxt = pick_next(logits, **pick_kwargs)
        cands = "  ".join(f"{tokenizer.decode([t])!r}:{p:.3f}" for t, p in zip(ti.tolist(), tp.tolist()))
        print(f"step {i}: {cands}   -> chose {tokenizer.decode([nxt])!r}")
        prompt += tokenizer.decode([nxt])
        out = llm(input_ids=torch.tensor([[nxt]], device=llm.device), past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[0, -1]
    print("\n" + prompt)

torch.manual_seed(0)
show_steps("Moscow is the capital of", steps=10)

### How temperature reshapes the distribution

In [ ]:
@torch.no_grad()
def next_token_logits(prompt):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(llm.device)
    return llm(input_ids=ids).logits[0, -1].float()

logits = next_token_logits("My favourite food is")
top_ids = torch.softmax(logits, -1).topk(20).indices
labels = [tokenizer.decode([t]) for t in top_ids.tolist()]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, T in zip(axes, (0.5, 1.0, 1.5)):
    probs = torch.softmax(logits / T, -1)
    ax.bar(range(20), probs[top_ids].cpu().numpy())
    ax.set_xticks(range(20)); ax.set_xticklabels(labels, rotation=75)
    n_nuc = (nucleus_filter(probs, 0.9) > 0).sum().item()
    ax.set_title(f"τ = {T}   (|nucleus p=0.9| = {n_nuc})")
plt.tight_layout(); plt.show()

### Task: nucleus sampling from scratch (don't use `model.generate`)

`nucleus_sampling` runs one forward pass and returns the sampled next token together with the full list of nucleus tokens, i.e. every token with non-zero probability after filtering.

The token is sampled **in proportion to its renormalized probability**, not uniformly within the nucleus.

In [ ]:
from typing import Tuple, List

@torch.no_grad()
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    """Sample the next token from the nucleus: the smallest set of tokens with total probability >= prob."""
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    probs = torch.softmax(model(input_ids=ids).logits[0, -1].float(), dim=-1)

    sorted_p, sorted_ix = torch.sort(probs, descending=True)
    n_keep = int(((torch.cumsum(sorted_p, 0) - sorted_p) < prob).sum())   # >= 1
    nucleus_p, nucleus_ix = sorted_p[:n_keep], sorted_ix[:n_keep]

    choice = nucleus_ix[torch.multinomial(nucleus_p / nucleus_p.sum(), 1)].item()
    sampled_token = tokenizer.decode([choice]).strip()
    possible_tokens = [tokenizer.decode([t]).strip() for t in nucleus_ix.tolist()]
    return sampled_token, possible_tokens

for test_prompt, p in [("Elbrus is the highest", 0.9), ("Large language models can learn to", 0.4)]:
    tok, cands = nucleus_sampling(llm, tokenizer, test_prompt, prob=p)
    print(f"{test_prompt!r} (p={p}) -> {tok!r}   nucleus ({len(cands)}): {cands}")
    assert tok in cands

# generic sanity checks (the exact nucleus depends on the checkpoint)
_, tiny = nucleus_sampling(llm, tokenizer, "Elbrus is the highest", prob=1e-6)
_, small = nucleus_sampling(llm, tokenizer, "Elbrus is the highest", prob=0.5)
_, large = nucleus_sampling(llm, tokenizer, "Elbrus is the highest", prob=0.95)
assert len(tiny) == 1 and set(small) <= set(large)
print("nucleus tests passed")

### Full generator from scratch

This puts the pieces together: a KV cache, the shared `pick_next` (temperature, top-k, top-p), and a stop at the EOS token. As a check, our **greedy** output should reproduce `generate(do_sample=False)` token for token. fp16 non-determinism can occasionally flip a near-tie.

In [ ]:
@torch.no_grad()
def llm_generate(prompt, max_new_tokens=60, **pick_kwargs):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(llm.device)
    out = llm(input_ids=ids, use_cache=True)
    past, logits = out.past_key_values, out.logits[0, -1]
    new = []
    for _ in range(max_new_tokens):
        nxt = pick_next(logits, **pick_kwargs)
        if nxt == tokenizer.eos_token_id:
            break
        new.append(nxt)
        out = llm(input_ids=torch.tensor([[nxt]], device=llm.device), past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[0, -1]
    return new

prompt = "Elbrus is the highest"
ours = llm_generate(prompt, max_new_tokens=30, temperature=0)
batch = tokenizer(prompt, return_tensors="pt").to(llm.device)
hf = llm.generate(**batch, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)[0, batch.input_ids.shape[1]:]
print("ours:", tokenizer.decode(ours))
print("HF  :", tokenizer.decode(hf))
print("greedy matches HF generate:", ours == hf.tolist()[:len(ours)])

torch.manual_seed(0)
for kw in (dict(temperature=0.7), dict(top_k=20), dict(top_p=0.9), dict(top_p=0.5)):
    print(f"\n--- {kw}\n{prompt}{tokenizer.decode(llm_generate(prompt, 40, **kw))}")

### Bonus: beam search from scratch on Llama

This is the same algorithm as the char-RNN version. To keep it short we re-run the full prefix for each beam (batched) instead of reordering the KV cache.

In [ ]:
@torch.no_grad()
def llm_beam_search(prompt, beam_size=4, max_new_tokens=20, length_penalty=1.0):
    eos = tokenizer.eos_token_id
    start = tokenizer(prompt, return_tensors="pt").input_ids[0].tolist()
    beams = [(start, 0.0, False)]                  # (ids, logprob, finished)
    for _ in range(max_new_tokens):
        live = [b for b in beams if not b[2]]
        if not live:
            break
        ids = torch.tensor([b[0] for b in live], device=llm.device)   # all live beams have equal length
        logp = torch.log_softmax(llm(input_ids=ids).logits[:, -1].float(), -1)
        top_lp, top_ix = logp.topk(beam_size, dim=-1)
        cands = [b for b in beams if b[2]]
        for (seq, s, _), lps, ixs in zip(live, top_lp.tolist(), top_ix.tolist()):
            cands += [(seq + [t], s + lp, t == eos) for lp, t in zip(lps, ixs)]
        norm = lambda b: b[1] / (len(b[0]) - len(start)) ** length_penalty
        beams = sorted(cands, key=norm, reverse=True)[:beam_size]   # live beams all grow by 1/step -> same length
    return [(tokenizer.decode(b[0][len(start):], skip_special_tokens=True), b[1]) for b in beams]

for k in (1, 4, 8):
    text_, lp = llm_beam_search("The best thing about deep learning is", beam_size=k)[0]
    print(f"beam {k} (log p {lp:.2f}): {text_!r}")

# Char-RNN vs Llama: bits per character on the same text

Perplexities over different vocabularies can't be compared directly. Per-character cross-entropy can: sum Llama's NLL over all tokens of a speech, then divide by the number of **characters**.

Caveats: Llama has almost certainly seen Shakespeare during pre-training, and the char-RNN is also charged for predicting EOS. Read this as an illustration of scale, not a benchmark.

In [ ]:
@torch.no_grad()
def llm_bits_per_char(texts):
    total_nll, total_chars = 0.0, 0
    for t in texts:
        ids = tokenizer(t, return_tensors="pt").input_ids.to(llm.device)   # includes <|begin_of_text|>
        logits = llm(input_ids=ids).logits[0, :-1].float()
        total_nll += F.cross_entropy(logits, ids[0, 1:], reduction="sum").item()
        total_chars += len(t)
    return total_nll / total_chars / math.log(2)

subset = dev_lines[:300]
rnn_bpc = score_lines(model, subset) / math.log(2)
llm_bpc = llm_bits_per_char([show(l) for l in subset])
print(f"char-RNN ({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params): {rnn_bpc:.3f} bits/char")
print(f"{model_name} ({sum(p.numel() for p in llm.parameters()) / 1e9:.1f}B params): {llm_bpc:.3f} bits/char")

### Things to try

1. Swap `nn.LSTM` for `nn.GRU`, or vary `hid_size` and `num_layers`. Plot dev bpc against parameter count.
2. Train the char-RNN on a different public corpus, such as the [arXiv abstracts](https://www.kaggle.com/datasets/neelshah18/arxivdataset) from the course homework, or any Project Gutenberg book.
3. Ask Llama to *continue a Shakespeare speech*. Compare its nucleus sizes, character by character, with the char-RNN plot above.
4. Implement **min-p** sampling (keep tokens with $p_i \ge \alpha\cdot\max_j p_j$) and compare it with top-p at high temperature.
5. Make `llm_beam_search` reuse the KV cache by reordering `past_key_values` with the selected beam indices, as `beam_search` does with the LSTM state.